# S6.1 · 条件价值曲线

「联邦学习有没有用」是个坏问题——它**在什么条件下有用**才是可回答的。

本步扫描四个结构参数，画出增益随条件变化的曲线，并给出**盈亏平衡点**：增益低于多少时，项目不值得做。

In [1]:
ROUND_DP = 4          # 表格展示精度（不影响任何计算结果）
import sys, subprocess, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "registry").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, yaml
CONFIG_PATH = ROOT / "modules/m5_modeling/configs/experiment.yaml"
config = yaml.safe_load(open(CONFIG_PATH, encoding="utf-8"))
seed = config.get("seeds", [config.get("seed")])[0]
git = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, cwd=ROOT).stdout.strip()
print("config:", CONFIG_PATH.relative_to(ROOT))
print("seed  :", seed, "| 全部种子:", config.get("seeds"))
print("git   :", git or "(未提交)")
print("numpy :", np.__version__, "| pandas:", pd.__version__)

config: modules/m5_modeling/configs/experiment.yaml
seed  : 11 | 全部种子: [11, 22, 33, 44, 55]
git   : 7f29b9e
numpy : 2.3.5 | pandas: 2.3.3


In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "Heiti TC", "PingFang SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
FIGSIZE_WIDE = (9, 4)
FIG_DPI = 150
GRID_W = 11
GRID_H = 7
FIGDIR = ROOT / "modules/m6_evaluation" / "results"
sw = pd.read_csv(ROOT / 'modules/m5_modeling/results/sweep_results.csv')
L3, L1, L0 = 'L3a_联邦LR', 'L1_加k匿名统计_LR', 'L0_内地单方_LR'
def curve(param):
    d = sw[sw.sweep_param == param]
    t = d.pivot_table(index='sweep_value', columns='level', values='auc', aggfunc='mean')
    return pd.DataFrame({'L0': t[L0], 'L1': t[L1], 'L3a': t[L3],
                         'L3a减L0': t[L3]-t[L0], 'L3a减L1': t[L3]-t[L1]})
curve('complementarity').round(ROUND_DP)

,L0,L1,L3a,L3a减L0,L3a减L1
sweep_value,,,,,
0.0,0.7215,0.7191,0.7469,0.0254,0.0278
0.4,0.7230,0.7210,0.7584,0.0354,0.0375
0.8,0.7089,0.7068,0.7868,0.0780,0.0800
1.2,0.7075,0.7039,0.8305,0.1230,0.1266
1.6,0.7053,0.7050,0.8823,0.1770,0.1774
2.0,0.6922,0.6937,0.9011,0.2089,0.2074


**互补性 λ 是主导因子**：λ=0 时增益 0.025，λ=2.0 时增益 0.209——相差八倍。

In [3]:
PARAMS = ['complementarity', 'overlap_rate', 'redundancy', 'match_error_rate']
LABELS = ['互补性 λ', '重叠率 ρ', '冗余度', '匹配错误率']
N_COL = 2
fig, axes = plt.subplots(N_COL, N_COL, figsize=(GRID_W, GRID_H))
for ax, p, lab in zip(axes.ravel(), PARAMS, LABELS):
    c = curve(p)
    ax.plot(c.index, c['L3a减L0'], marker='o', label='L3a − L0')
    ax.plot(c.index, c['L3a减L1'], marker='s', label='L3a − L1')
    ax.axhline(0, color='gray', linewidth=1)
    ax.set_title(lab); ax.set_xlabel(lab); ax.set_ylabel('AUC 增益'); ax.legend()
fig.suptitle('条件价值曲线：纵向联邦的增益在什么条件下成立')
fig.tight_layout(); fig.savefig(FIGDIR / 'conditional_value_curves.png', dpi=FIG_DPI)
plt.close(fig); print('图已保存 conditional_value_curves.png')

图已保存 conditional_value_curves.png


## 关键对照：40 种子下的 L1 vs L3

这是本项目**最核心的一个数字**。为压窄置信区间，此对照单独用 40 个种子。

「安慰剂臂」把 B 侧换成同形状纯噪声——它分离出「分段本身带来的灵活性」，剩下的才是 B 侧数据的**净贡献**。

In [4]:
crit = pd.read_csv(ROOT / 'modules/m5_modeling/results/l1_vs_l3_critical.csv')
CI_Z = 1.96
def ci(x):
    m = x.mean(); se = x.std(ddof=1)/np.sqrt(len(x))
    return pd.Series({'均值': m, 'CI下界': m-CI_Z*se, 'CI上界': m+CI_Z*se})
cmp = pd.DataFrame({
    'L1减L0（表观增益）': ci(crit.L1_真实 - crit.L0),
    'L1减安慰剂（B的净贡献）': ci(crit.L1_真实 - crit.L1_安慰剂),
    'L3a减L0（VFL总增益）': ci(crit.L3a - crit.L0),
    'L3a减L1（VFL净增量）': ci(crit.L3a - crit.L1_真实)}).T
cmp['显著'] = np.where(cmp['CI下界'] > 0, '是', '否')
cmp.round(ROUND_DP)

,均值,CI下界,CI上界,显著
L1减L0（表观增益）,0.0041,0.0004,0.0078,是
L1减安慰剂（B的净贡献）,0.0078,0.0039,0.0116,是
L3a减L0（VFL总增益）,0.0631,0.0537,0.0726,是
L3a减L1（VFL净增量）,0.0590,0.0501,0.0680,是


In [5]:
share = (crit.L1_真实 - crit.L1_安慰剂).mean() / (crit.L3a - crit.L0).mean()
PCT = 100
print(f'L1 捕获了 VFL 价值的 {share*PCT:.1f}%')
print('→ L1 是真实的竞争者，但拿不到大头；VFL 的商业前提成立')

L1 捕获了 VFL 价值的 12.3%
→ L1 是真实的竞争者，但拿不到大头；VFL 的商业前提成立


## 盈亏平衡：增益要多大才值得做

决策价值 ≈ **增益 × 可触达人数 × 单人价值**。低重叠场景下单人增益虽高，但可触达人数极小，总价值反而低。

In [6]:
ov = curve('overlap_rate')
BASE_N = 1_000_000
ov['可触达人数'] = (ov.index * BASE_N).astype(int)
ov['相对总价值'] = ov['L3a减L1'] * ov['可触达人数']
ov[['L3a减L1', '可触达人数', '相对总价值']].round(ROUND_DP)

,L3a减L1,可触达人数,相对总价值
sweep_value,,,
0.05,0.1019,50000,5095.4194
0.10,0.0762,100000,7623.2810
0.20,0.0718,200000,14360.6938
0.30,0.0800,300000,24002.5862
0.50,0.0822,500000,41106.5030


**结论**：单人增益与可触达人数方向相反，总价值在中高重叠区达到最大。只看 AUC 增益会得出「越低重叠越好」的错误结论。